# Hidden common causes (BottomUpParceLiNGAM)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/bottom_up_parce_lingam.ipynb)

BottomUpParceLiNGAM assumes linear mechanisms with non-Gaussian noise and finds a causal order from
the bottom up: at each step it looks for a sink, a variable whose regression residual on the others
is independent of them. A hidden common cause leaves the variables it confounds without a sink, so
the method stops there and leaves them unordered, joined by `o-o`, rather than guess an order.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.4.0` from PyPI;
local kernels use the installed environment.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.4.0" pandas

## Data

A linear model with uniform noise over five variables, one of them hidden: `h` drives `x0` and
`x1`, `x0` drives `x3`, and `x1` drives `x2`. The data keep only `x0` to `x3`, so `x0` and `x1`
depend on each other with no edge between them. The true graph over the observed variables marks
that pair `<->`.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.viz
from andrey.core import ARROW, TAIL

COLUMNS = ["x0", "x1", "x2", "x3"]


def sample(seed: int, n: int = 2000) -> pd.DataFrame:
    """Draw n rows of x0..x3; the hidden h is left out."""
    rng = np.random.default_rng(seed)
    h, e0, e1, e2, e3 = rng.uniform(-1, 1, (5, n))
    x0, x1 = 1.5 * h + e0, 1.5 * h + e1
    return pd.DataFrame({"x0": x0, "x1": x1, "x2": 0.6 * x1 + e2, "x3": 0.6 * x0 + e3})


# marks[i, j] is the mark at x_i of the edge between x_i and x_j.
marks = np.zeros((4, 4), dtype=np.int8)
marks[0, 1] = marks[1, 0] = ARROW  # x0 <-> x1
marks[0, 3], marks[3, 0] = TAIL, ARROW  # x0 -> x3
marks[1, 2], marks[2, 1] = TAIL, ARROW  # x1 -> x2
truth = andrey.GraphStructure.from_numpy(marks, kind="pag", labels=tuple(COLUMNS))

X = sample(seed=0)
print(f"{X.shape[0]} observations, {X.shape[1]} variables")
display(andrey.viz.draw(truth))

## Fit BottomUpParceLiNGAM

`andrey.bottom_up_parce_lingam` returns a PAG: `->` for a direct cause and `o-o` between variables
it leaves unordered. `metadata["causal_order"]` gives the order, the unordered variables nested in
one list. `alpha` is the significance level of the sink test, Bonferroni-corrected across the
remaining variables; at the default 0.1 a true sink is sometimes rejected by chance on this small
model, so this notebook uses 0.01.

In [ ]:
out = andrey.bottom_up_parce_lingam(X, alpha=0.01)
print("causal order:", out.metadata["causal_order"])
for source, target, kind in out.structure.oriented_edges():
    print(f"{source} {'->' if kind == 'directed' else 'o-o'} {target}")
display(andrey.viz.draw(out.structure))

## Edge weights

`weighted_adjacency[i, j]` is the coefficient of variable `i` in the equation for variable `j`,
from an adaptive-Lasso regression of each variable on those before it in the order.

In [ ]:
pd.DataFrame(out.weighted_adjacency, index=COLUMNS, columns=COLUMNS).round(2)

## Check over ten datasets

The same model, drawn with ten seeds. For each, the table says whether the method left exactly
`x0` and `x1` unordered, whether it found both direct edges, and its causal order.

In [ ]:
rows = {}
for seed in range(10):
    fit = andrey.bottom_up_parce_lingam(sample(seed), alpha=0.01)
    order = fit.metadata["causal_order"]
    blocks = [set(item) for item in order if isinstance(item, list)]
    found = set(fit.structure.oriented_edges(index=True))
    rows[seed] = {
        "x0, x1 unordered": blocks == [{0, 1}],
        "direct edges": {(0, 3, "directed"), (1, 2, "directed")} <= found,
        "causal order": str(order),
    }
pd.DataFrame(rows).T.rename_axis("seed")

## When to use it

BottomUpParceLiNGAM relies on linear mechanisms and non-Gaussian noise. Its unordered block holds
the confounded variables and every variable above them in the order, so on a larger graph it can
join pairs that share no cause, and edges inside the block stay unoriented. RCD marks only the
pairs it attributes to a hidden cause; without hidden causes, DirectLiNGAM returns a full DAG. The
[0.2.0 benchmarks](https://andrey.abel.ai/docs/benchmarks/0.2.0.html) measure it on larger
graphs. Change the seed, `alpha`, or `n` and rerun to see how it responds.